# **THIẾT LẬP MÔI TRƯỜNG**

1. LẤY METADATA CỦA ARXIV TỪ KAGGLE

In [ ]:
import os
import json

KAGGLE_USERNAME = "gianggg"
KAGGLE_KEY = "04c87cdc5f61ab3af98c5c480a575714"

kaggle_credentials = {
    "username": KAGGLE_USERNAME,
    "key": KAGGLE_KEY
}

os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("✅ Đã thiết lập xong Chìa khóa Kaggle!")

✅ Đã thiết lập xong Chìa khóa Kaggle!


2. KÉO METADATA VỀ DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/BigDataPTIT2026/Raw_Data"
!mkdir -p {DRIVE_PATH}

print("🚀 Đang kéo dữ liệu 4GB từ Kaggle về Drive...")
!kaggle datasets download -d Cornell-University/arxiv -p {DRIVE_PATH}

Mounted at /content/drive
🚀 Đang kéo dữ liệu 4GB từ Kaggle về Drive...
Dataset URL: https://www.kaggle.com/datasets/Cornell-University/arxiv
License(s): CC0-1.0
100% 1.61G/1.61G [00:39<00:00, 43.5MB/s]



3. CONNECT GCS

In [ ]:
from google.colab import auth
from google.cloud import storage

auth.authenticate_user()

project_id = 'bigdataptit2026'
bucket_name = 'bigdata-n9-ptit-final'
client = storage.Client(project=project_id)
bucket = client.bucket(bucket_name)

4. TẢI THƯ VIỆN CẦN THIỂT

In [ ]:
!pip install transformers torch faiss-cpu pandas pyarrow gcsfs PyMuPDF python-Levenshtein datasketch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.6 MB/s eta 0:00:00


# **THU THẬP DỮ LIỆU**

1. CHUẨN HÓA METADATA THU ĐƯỢC VÀ LƯU VÀO TẦNG BRONE TRONG GCS

In [ ]:
import zipfile
import json
import hashlib
from datetime import datetime
import os

# 1. Cấu hình đường dẫn
RAW_ZIP_PATH = "/content/drive/MyDrive/BigDataPTIT2026/Raw_Data/arxiv.zip"
OUTPUT_DIR = "/content/drive/MyDrive/BigDataPTIT2026/Normalized_Data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "arxiv_metadata_normalized.jsonl")

# Tạo thư mục đầu ra nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_hash(text):
    """Tạo mã băm SHA-256 từ chuỗi đầu vào"""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

print("🚀 Bắt đầu quá trình chuẩn hóa sang cấu trúc Flat...")

# 2. Mở file ZIP và xử lý dòng theo dòng (Stream Processing)
with zipfile.ZipFile(RAW_ZIP_PATH, 'r') as z:
    json_filename = z.namelist()[0] # Thường là 'arxiv-metadata-oai-snapshot.json'

    with z.open(json_filename) as f_in, open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        for i, line in enumerate(f_in):
            # Đọc dữ liệu thô từ 1 dòng của Kaggle
            raw_data = json.loads(line)

            # --- TẠO CÁC BIẾN CƠ SỞ ---
            # ID của arxiv đôi khi có dấu gạch chéo ở các bài báo cũ (vd: hep-ph/0704.0001),
            # ta thay thế bằng gạch dưới để làm tên file an toàn
            safe_id = raw_data['id'].replace('/', '_')
            doc_id = f"arxiv_{safe_id}"
            source_url = f"https://arxiv.org/abs/{raw_data['id']}"
            pdf_url = f"https://arxiv.org/pdf/{raw_data['id']}.pdf"
            hash_val = generate_hash(source_url)

            # Xử lý mảng tác giả: ghép (Firstname + Lastname) từ mảng authors_parsed của Kaggle
            # a[1] là Tên, a[0] là Họ
            clean_authors = [f"{a[1]} {a[0]}".strip() for a in raw_data.get('authors_parsed', [])]

            #
            metadata = {
                "doc_id": doc_id,
                "hash_sha256": hash_val,
                "title": raw_data.get('title', '').replace('\n', ' ').strip(),
                "abstract": raw_data.get('abstract', '').replace('\n', ' ').strip(),
                "authors": clean_authors,
                "language": "en",
                "categories": raw_data.get('categories', '').split(' '),
                "publish_date": raw_data.get('update_date', ''),
                "doi": raw_data.get('doi', ''),
                "source_url": source_url,
                "pdf_url": pdf_url,
                "local_path_bronze": f"gs://bigdata-n9-ptit/bronze/arxiv/{doc_id}.pdf",
                "scraped_at": datetime.utcnow().isoformat() + "Z"
            }

            # 4. Ghi đối tượng phẳng này thành 1 dòng JSON vào file đầu ra
            f_out.write(json.dumps(metadata, ensure_ascii=False) + '\n')

            # Theo dõi tiến độ
            if i % 100000 == 0 and i > 0:
                print(f"✅ Đã xử lý {i} bài báo...")

print(f"🏁 Hoàn thành! Dữ liệu phẳng đã được lưu tại: {OUTPUT_FILE}")

🚀 Bắt đầu quá trình chuẩn hóa sang cấu trúc Flat...


/tmp/ipykernel_19801/277386047.py:57: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at": datetime.utcnow().isoformat() + "Z"


✅ Đã xử lý 100000 bài báo...
✅ Đã xử lý 200000 bài báo...
✅ Đã xử lý 300000 bài báo...
✅ Đã xử lý 400000 bài báo...
✅ Đã xử lý 500000 bài báo...
✅ Đã xử lý 600000 bài báo...
✅ Đã xử lý 700000 bài báo...
✅ Đã xử lý 800000 bài báo...
✅ Đã xử lý 900000 bài báo...
✅ Đã xử lý 1000000 bài báo...
✅ Đã xử lý 1100000 bài báo...
✅ Đã xử lý 1200000 bài báo...
✅ Đã xử lý 1300000 bài báo...
✅ Đã xử lý 1400000 bài báo...
✅ Đã xử lý 1500000 bài báo...
✅ Đã xử lý 1600000 bài báo...
✅ Đã xử lý 1700000 bài báo...
✅ Đã xử lý 1800000 bài báo...
✅ Đã xử lý 1900000 bài báo...
✅ Đã xử lý 2000000 bài báo...
✅ Đã xử lý 2100000 bài báo...
✅ Đã xử lý 2200000 bài báo...
✅ Đã xử lý 2300000 bài báo...
✅ Đã xử lý 2400000 bài báo...
✅ Đã xử lý 2500000 bài báo...
✅ Đã xử lý 2600000 bài báo...
✅ Đã xử lý 2700000 bài báo...
✅ Đã xử lý 2800000 bài báo...
✅ Đã xử lý 2900000 bài báo...
✅ Đã xử lý 3000000 bài báo...
🏁 Hoàn thành! Dữ liệu phẳng đã được lưu tại: /content/drive/MyDrive/BigDataPTIT2026/Normalized_Data/arxiv_met

In [ ]:
# 1. Khai báo biến đường dẫn
INPUT_JSONL="/content/drive/MyDrive/BigDataPTIT2026/Normalized_Data/arxiv_metadata_normalized.jsonl"
DEST_GCS="gs://bigdata-n9-ptit-final/bronze/raw_metadata/arxiv/2026_04_30/arxiv_full_dump.jsonl"

# 2. Lệnh copy siêu tốc thế hệ mới của Google Cloud
!gcloud storage cp $INPUT_JSONL $DEST_GCS

print("✅ Đã bay lên Cloud thành công!")

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file:///content/drive/MyDrive/BigDataPTIT2026/Normalized_Data/arxiv_metadata_normalized.jsonl to gs://bigdata-n9-ptit/bronze/raw_metadata/arxiv/2026_04_30/arxiv_full_dump.jsonl

Average throughput: 72.8MiB/s
✅ Đã bay lên Cloud thành công!


2. CÀO PDF VỀ VÀO LƯU VÀO TRONG TẦNG BRONE

In [ ]:
import json
import requests
import time
from datetime import datetime
from google.cloud import storage
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import random

PROJECT_ID = 'bigdataptit2026'
BUCKET_NAME = 'bigdata-n9-ptit-final'

SOURCE_JSONL_PATH = "bronze/raw_metadata/arxiv/2026_04_30/arxiv_full_dump.jsonl"

# PHÂN CÔNG CÔNG VIỆC
START_INDEX = 13765
END_INDEX   = 23000

# Khởi tạo GCS Client
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)
today_str = datetime.now().strftime('%Y_%m_%d')

# Thiết lập Session HTTP
session = requests.Session()
retry = Retry(connect=5, backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

print(f" KHỞI ĐỘNG BỘ CÀO PDF (Task: Dòng {START_INDEX} -> {END_INDEX})...")
print(f" Nguồn đọc Metadata: gs://{BUCKET_NAME}/{SOURCE_JSONL_PATH}")
print(f" Đích lưu PDF: gs://{BUCKET_NAME}/bronze/raw_pdfs/arxiv/{today_str}/")
print("-" * 60)

# --- 2. HÀM TẢI VÀ ĐẨY THẲNG LÊN GCS
def download_pdf_to_gcs(doc_id, pdf_url):
    """Trả về 1 trong 3 trạng thái: 'SKIPPED', 'SUCCESS', 'ERROR'"""
    destination_blob_name = f"bronze/raw_pdfs/arxiv/{today_str}/{doc_id}.pdf"
    blob = bucket.blob(destination_blob_name)

    # KIỂM TRA CHỐNG TRÙNG LẶP (RESUME)
    if blob.exists():
        return "SKIPPED"

    try:
        response = session.get(pdf_url, headers=headers, timeout=30)

        if response.status_code == 200 and 'application/pdf' in response.headers.get('Content-Type', ''):
            blob.upload_from_string(response.content, content_type='application/pdf')
            return "SUCCESS"
        else:
            print(f"\n Lỗi tải {doc_id}: HTTP {response.status_code}")
            return "ERROR"

    except Exception as e:
        print(f"\n Văng lỗi mạng tại {doc_id}: {e}")
        return "ERROR"

# 3.ĐỌC LUỒNG TỪ GCS
success_count = 0
skipped_count = 0
error_count = 0
total_task = END_INDEX - START_INDEX

source_blob = bucket.blob(SOURCE_JSONL_PATH)

print("\n Đang kết nối tới file trên GCS...")
with source_blob.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < START_INDEX:
            continue
        if i >= END_INDEX:
            print(f"\n Đã hoàn thành khối lượng công việc được giao (Đến dòng {END_INDEX}).")
            break

        metadata = json.loads(line)
        doc_id = metadata['doc_id']
        pdf_url = metadata.get('pdf_url', '')

        if not pdf_url:
            continue

        # Tiến hành xử lý tải
        status = download_pdf_to_gcs(doc_id, pdf_url)

        current_relative_idx = i - START_INDEX + 1

        # Cập nhật thống kê và in tiến độ
        if status == "SKIPPED":
            skipped_count += 1
            # In đè lên cùng 1 dòng để màn hình log đỡ bị trôi
            print(f"\r⏭ [Tiến độ: {current_relative_idx}/{total_task}] - Bỏ qua: {doc_id} (Đã có sẵn)", end="", flush=True)

        elif status == "SUCCESS":
            success_count += 1
            print(f"\n [Tiến độ: {current_relative_idx}/{total_task}] - Đã tải mới: {doc_id}")
            delay = random.uniform(1.5, 2.5)
            time.sleep(delay)
        elif status == "ERROR":
            error_count += 1
            print(f" [Tiến độ: {current_relative_idx}/{total_task}] - Thất bại: {doc_id}")
            delay = random.uniform(1.5, 2.5)
            time.sleep(delay)

# --- 4. TỔNG KẾT BÁO CÁO ---
print("\n\n" + "="*40)
print(" BÁO CÁO TIẾN ĐỘ TASK")
print("="*40)
print(f"🔹 Dải dữ liệu: Từ {START_INDEX} đến {END_INDEX}")
print(f" Tải mới thành công : {success_count} files")
print(f" Bỏ qua (Đã có sẵn): {skipped_count} files")
print(f" Lỗi / Thất bại     : {error_count} files")
print("="*40)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
 [Tiến độ: 4789/16515] - Đã tải mới: arxiv_0706.2587

 [Tiến độ: 4790/16515] - Đã tải mới: arxiv_0706.2588

 [Tiến độ: 4791/16515] - Đã tải mới: arxiv_0706.2589

 [Tiến độ: 4792/16515] - Đã tải mới: arxiv_0706.2590

 [Tiến độ: 4793/16515] - Đã tải mới: arxiv_0706.2591

 [Tiến độ: 4794/16515] - Đã tải mới: arxiv_0706.2592

 [Tiến độ: 4795/16515] - Đã tải mới: arxiv_0706.2593

 [Tiến độ: 4796/16515] - Đã tải mới: arxiv_0706.2594

 [Tiến độ: 4797/16515] - Đã tải mới: arxiv_0706.2595

 [Tiến độ: 4798/16515] - Đã tải mới: arxiv_0706.2596

 [Tiến độ: 4799/16515] - Đã tải mới: arxiv_0706.2597

 [Tiến độ: 4800/16515] - Đã tải mới: arxiv_0706.2598

 [Tiến độ: 4801/16515] - Đã tải mới: arxiv_0706.2599

 [Tiến độ: 4802/16515] - Đã tải mới: arxiv_0706.2600

 [Tiến độ: 4803/16515] - Đã tải mới: arxiv_0706.2601

 [Tiến độ: 4804/16515] - Đã tải mới: arxiv_0706.2602

 [Tiến độ: 4805/16515] - Đã tải mới: arxiv_0706.2603

 [Tiến độ: 4806/16515] - 

KeyboardInterrupt: 

CÀO TEXT TỪ PDF VỀ VÀ LƯU Ở TẦNG SILVER

In [ ]:
"""
01_extract_text.py - Boc text tu PDF (Toi uu cho 300GB+)
Chay tren: Colab
Input:  gs://BUCKET/bronze/raw_pdfs/arxiv/
Output: gs://BUCKET/silver/arxiv_text_parquet/

Chien luoc:
  - Download tung batch ~10000 files ve local (tranh tran disk)
  - Extract bang ProcessPool (CPU-bound, nhanh hon ThreadPool)
  - Checkpoint theo file_path (khong trung/khong sot khi crash)
  - Tu dong don dep disk sau moi batch
"""

import fitz
import pyarrow as pa
import pyarrow.parquet as pq
from concurrent.futures import ProcessPoolExecutor
from google.cloud import storage
from io import BytesIO
import os, glob, time, shutil

# === CAU HINH ===
PROJECT = "bigdataptit2026"
BUCKET_NAME = "bigdata-n9-ptit-final"
PDF_PREFIX = "bronze/raw_pdfs/arxiv/"
OUTPUT_PREFIX = "silver/arxiv_text_parquet/"
LOCAL_DIR = "/tmp/pdfs_batch/"
DOWNLOAD_BATCH = 10000   # So file download moi lan (tranh tran disk)
PARQUET_BATCH = 5000     # So file moi parquet part
MAX_WORKERS = os.cpu_count()      # ProcessPool workers (= so CPU Colab)

fitz.TOOLS.mupdf_display_errors(False)
client = storage.Client(project=PROJECT)
bucket = client.bucket(BUCKET_NAME)

# === [1/4] QUET DANH SACH PDF ===
print("[1/4] Quet danh sach PDF...")
t0 = time.time()
all_blobs = list(bucket.list_blobs(prefix=PDF_PREFIX))
pdf_blobs = [b for b in all_blobs if b.name.endswith(".pdf")]
total = len(pdf_blobs)
print("  Tong: {:,} PDF files".format(total))

# === [2/4] CHECKPOINT: Doc file_path da xu ly ===
print("[2/4] Doc checkpoint...")
processed_paths = set()
existing_parts = list(bucket.list_blobs(prefix=OUTPUT_PREFIX))
parquet_blobs = [b for b in existing_parts if b.name.endswith(".parquet")]

if parquet_blobs:
    print("  Doc {} parquet files de lay file_path da xu ly...".format(len(parquet_blobs)))
    for i, blob in enumerate(parquet_blobs):
        try:
            buf = BytesIO(blob.download_as_bytes())
            table = pq.read_table(buf, columns=["file_path"])
            for fp in table.column("file_path").to_pylist():
                processed_paths.add(fp)
        except:
            pass
        if (i + 1) % 50 == 0:
            print("    {}/{} parquet files".format(i + 1, len(parquet_blobs)))

print("  Da xu ly: {:,} files".format(len(processed_paths)))

# Loc bo PDF da xu ly
pending_blobs = []
for b in pdf_blobs:
    fp = "gs://{}/{}".format(BUCKET_NAME, b.name)
    if fp not in processed_paths:
        pending_blobs.append(b)

print("  Con lai: {:,} files can xu ly".format(len(pending_blobs)))
print("  Checkpoint mat: {:.0f}s".format(time.time() - t0))

if not pending_blobs:
    print("\nKhong con file nao can xu ly!")
    exit()

# === HAM EXTRACT (chay trong ProcessPool) ===
def extract_one(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        text = "\n".join(page.get_text() for page in doc).strip()
        doc.close()
        wc = len(text.split())
        if wc <= 10:
            return None
        return {
            "filename": os.path.basename(pdf_path),
            "text": text,
            "word_count": wc,
        }
    except:
        return None

# === [3/4] XU LY TUNG DOWNLOAD BATCH ===
print("\n[3/4] Bat dau xu ly...")
t1 = time.time()
part_counter = len(parquet_blobs)  # Tiep tuc danh so tu parquet cuoi
total_extracted = 0
total_pending = len(pending_blobs)

for dl_start in range(0, total_pending, DOWNLOAD_BATCH):
    dl_batch = pending_blobs[dl_start:dl_start + DOWNLOAD_BATCH]
    dl_end = min(dl_start + DOWNLOAD_BATCH, total_pending)

    # Don dep + tao thu muc
    if os.path.exists(LOCAL_DIR):
        shutil.rmtree(LOCAL_DIR)
    os.makedirs(LOCAL_DIR, exist_ok=True)

    # Download batch nay ve local
    print("\n  --- Download batch {}-{}/{} ---".format(dl_start, dl_end, total_pending))
    td = time.time()

    # Ghi danh sach file can download
    with open("/tmp/download_list.txt", "w") as f:
        for b in dl_batch:
            f.write("gs://{}/{}\n".format(BUCKET_NAME, b.name))

    os.system("cat /tmp/download_list.txt | gsutil -m cp -I {}".format(LOCAL_DIR))
    print("  Download: {:.0f}s".format(time.time() - td))

    # Tao mapping filename -> gcs_path
    path_map = {}
    for b in dl_batch:
        filename = b.name.split("/")[-1]
        path_map[filename] = "gs://{}/{}".format(BUCKET_NAME, b.name)

    # Extract text tu local files
    local_pdfs = sorted(glob.glob(LOCAL_DIR + "*.pdf"))
    print("  Extract {} local files...".format(len(local_pdfs)))

    for pq_start in range(0, len(local_pdfs), PARQUET_BATCH):
        pq_batch = local_pdfs[pq_start:pq_start + PARQUET_BATCH]

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
            results = [r for r in ex.map(extract_one, pq_batch) if r]

        if results:
            # Map filename -> gcs file_path
            rows = []
            for r in results:
                gcs_path = path_map.get(r["filename"], "")
                if gcs_path:
                    rows.append({
                        "file_path": gcs_path,
                        "text": r["text"],
                        "word_count": r["word_count"],
                    })

            if rows:
                table = pa.table({
                    "file_path": [r["file_path"] for r in rows],
                    "text": [r["text"] for r in rows],
                    "word_count": [r["word_count"] for r in rows],
                })
                buf = BytesIO()
                pq.write_table(table, buf)
                buf.seek(0)
                part_name = "{}part-{:05d}.parquet".format(OUTPUT_PREFIX, part_counter)
                bucket.blob(part_name).upload_from_file(buf, content_type="application/octet-stream")
                part_counter += 1
                total_extracted += len(rows)

    # Don dep disk
    shutil.rmtree(LOCAL_DIR, ignore_errors=True)

    # Tien do
    elapsed = time.time() - t1
    done = min(dl_end, total_pending)
    speed = done / elapsed if elapsed > 0 else 0
    eta = (total_pending - done) / speed / 60 if speed > 0 else 0
    print("  Tien do: {:,}/{:,} ({:.1f}%) | {:.0f} files/s | ETA: {:.0f} phut | Extracted: {:,}".format(
        done, total_pending, 100.0 * done / total_pending, speed, eta, total_extracted))

# === [4/4] TONG KET ===
print("\n[4/4] Don dep...")
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
os.remove("/tmp/download_list.txt") if os.path.exists("/tmp/download_list.txt") else None

total_time = time.time() - t0
print("\n" + "=" * 60)
print("HOAN THANH!")
print("  Tong PDF:       {:,}".format(total))
print("  Da xu ly truoc: {:,}".format(len(processed_paths)))
print("  Xu ly moi:      {:,}".format(total_extracted))
print("  Thoi gian:      {:.0f} phut".format(total_time / 60))
print("  Output:         gs://{}/{}".format(BUCKET_NAME, OUTPUT_PREFIX))

Đang quét danh sách file từ Bronze...
Tổng: 1,000 PDF files
Đang kiểm tra checkpoint trong thư mục silver...
Bỏ qua 0 batches đã hoàn thành.
------------------------------------------------------------
 Tiến độ: |████████████████████| 100.00% (1,000/1,000) | Đã chạy: 4.3m | Còn lại: 0.0m

Done!


CLEAN RAW TEXT

In [ ]:
import re
import unicodedata
import pyarrow as pa
import pyarrow.parquet as pq
from google.cloud import storage
from io import BytesIO
import pandas as pd

PROJECT = "bigdataptit2026"
BUCKET_NAME = "bigdata-n9-ptit-final"
INPUT_PREFIX = "silver/arxiv_text_parquet/"
OUTPUT_PREFIX = "silver/arxiv_cleaned_parquet/"

client = storage.Client(project=PROJECT)
bucket = client.bucket(BUCKET_NAME)


def clean_arxiv_text(text):
    if not text or not isinstance(text, str):
        return {
            "clean_text": "", "has_references": False,
            "word_count_clean": 0, "reduction_ratio": 0.0, "quality_flag": "empty"
        }

    original_len = len(text.split())

    # Tang 1: Xoa ArXiv header
    text = re.sub(
        r'^arXiv:\S+\s+\[[\w.\-]+\]\s+\d+\s+\w+\s+\d{4}\s*\n',
        '', text, flags=re.MULTILINE)

    # Tang 2: Xoa journal template metadata
    text = re.sub(
        r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|'
        r'Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
        r'\s+\d{1,2},?\s+\d{4}\n\d{1,2}:\d{2}\n', '', text)
    text = re.sub(
        r'(?:WSPC|World Scientific|Typeset with|Preprint submitted to|'
        r'Preprint typeset using|Draft version)[^\n]*\n(?:[^\n]{0,60}\n){0,2}',
        '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[A-Z]{2,10}-[\w/\-]+(?:,\s*[A-Z]{2,10}-[\w/\-]+)*\s*\n',
                  '', text, flags=re.MULTILINE)

    # Tang 3: Xoa References (chi trong 40% cuoi bai)
    has_references = False
    total_len = len(text)
    search_start = int(total_len * 0.60)

    ref_match = re.search(
        r'\n\s*(?:References|REFERENCES|Bibliography|BIBLIOGRAPHY)\s*\n',
        text[search_start:])
    if ref_match:
        text = text[:search_start + ref_match.start()]
        has_references = True
    else:
        search_strict = int(total_len * 0.75)
        ref_match2 = re.search(
            r'\n(?:\[\d+\]|\d+[\)\.]\s)\s*[A-Z][^.]{15,}\n'
            r'(?:(?:\[\d+\]|\d+[\)\.]\s)\s*.{10,}\n){4,}',
            text[search_strict:])
        if ref_match2:
            text = text[:search_strict + ref_match2.start()]
            has_references = True

    # Tang 4: Xoa so trang
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)
    text = re.sub(r'\n\s*\d+/\d+\s*\n', '\n', text)
    text = re.sub(r'\bpage\s+\d+\s+of\s+\d+\b', '', text, flags=re.IGNORECASE)

    # Tang 5: Fix ligatures
    for lig, rep in {'\ufb00':'ff','\ufb01':'fi','\ufb02':'fl',
                     '\ufb03':'ffi','\ufb04':'ffl','\ufb05':'st','\ufb06':'st'}.items():
        text = text.replace(lig, rep)

    # Tang 6: Unicode NFKC
    text = unicodedata.normalize("NFKC", text)

    # Tang 7: Xoa ky tu khong in duoc
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # Tang 8: Fix hyphenation cuoi dong
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)

    # Tang 9: Xoa email, URL, chuan hoa whitespace
    text = re.sub(r'\[?\w+@[\w.]+\]?\(?mailto:[\w@.]+\)?', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = text.strip()

    # Quality flag
    clean_wc = len(text.split())
    ratio = clean_wc / original_len if original_len > 0 else 0
    if clean_wc < 300: qf = "empty"
    elif clean_wc < 500: qf = "low_quality"
    elif ratio < 0.3: qf = "over_cleaned"
    else: qf = "ok"

    return {
        "clean_text": text, "has_references": has_references,
        "word_count_clean": clean_wc, "reduction_ratio": round(ratio, 3),
        "quality_flag": qf,
    }


# === CHAY ===
blobs = [b for b in bucket.list_blobs(prefix=INPUT_PREFIX) if b.name.endswith(".parquet")]
print("Tong: {} files".format(len(blobs)))

existing = set(b.name for b in bucket.list_blobs(prefix=OUTPUT_PREFIX))
stats = {"total": 0, "ok": 0, "empty": 0}

for idx, blob in enumerate(blobs):
    out_name = blob.name.replace(INPUT_PREFIX, OUTPUT_PREFIX)
    if out_name in existing:
        continue

    df = pq.read_table(BytesIO(blob.download_as_bytes())).to_pandas()
    rows = []
    for _, row in df.iterrows():
        r = clean_arxiv_text(row["text"])
        rows.append({
            "file_path": row["file_path"],
            "clean_text": r["clean_text"],
            "word_count_raw": row.get("word_count", len(str(row["text"]).split())),
            "word_count_clean": r["word_count_clean"],
            "reduction_ratio": r["reduction_ratio"],
            "has_references": r["has_references"],
            "quality_flag": r["quality_flag"],
        })
        stats["total"] += 1
        stats[r["quality_flag"]] = stats.get(r["quality_flag"], 0) + 1

    out_buf = BytesIO()
    pq.write_table(pa.Table.from_pandas(pd.DataFrame(rows)), out_buf)
    out_buf.seek(0)
    bucket.blob(out_name).upload_from_file(out_buf, content_type="application/octet-stream")

    if (idx + 1) % 10 == 0 or idx == 0:
        print("  {}/{} | rows: {:,}".format(idx + 1, len(blobs), stats["total"]))

print("\nHoan thanh! {:,} bai | ok: {:,} | empty: {:,}".format(
    stats["total"], stats.get("ok", 0), stats.get("empty", 0)))

Tong: 2 files
  1/2 | rows: 500

Hoan thanh! 1,000 bai | ok: 999 | empty: 0


JOIN CLEAN TEXT, RAW TEXT VỚI METADATA ĐỂ TẠO RA DATA SILVER+ (CODE NÀY CHẠY TRÊN CỤM MÁY ẢO, KHÔNG CHẠY TRÊN COLAB)

In [ ]:
import re
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, udf, split, element_at, regexp_extract, regexp_replace
)
from pyspark.sql.types import StructType, StructField, StringType

BUCKET = "bigdata-n9-ptit-final"

spark = SparkSession.builder \
    .appName("SilverPlus") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.sql.autoBroadcastJoinThreshold", "50mb") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# === DOC 3 NGUON ===
print("[1/6] Doc du lieu...")
df_raw = spark.read.parquet("gs://{}/silver/arxiv_text_parquet/".format(BUCKET)) \
    .select("file_path", col("text").alias("raw_text"))

df_clean = spark.read.parquet("gs://{}/silver/arxiv_cleaned_parquet/".format(BUCKET))

df_meta = spark.read.json(
    "gs://{}/bronze/raw_metadata/arxiv/*/arxiv_full_dump.jsonl".format(BUCKET)
).select("doc_id", "title", "authors", "categories", col("abstract").alias("abstract"))

# === JOIN ===
print("[2/6] Join...")
df_joined = df_clean.join(df_raw, on="file_path", how="left")

df_joined = df_joined.withColumn(
    "join_key",
    regexp_replace(
        regexp_extract(col("file_path"), r"(arxiv_[^/]+)\.pdf$", 1),
        r"v\d+$", ""
    )
)

df_joined = df_joined.join(df_meta, df_joined.join_key == df_meta.doc_id, how="left")

total = df_joined.count()
has_title = df_joined.filter(col("title").isNotNull()).count()
print("  Tong: {} | Co title: {} | NULL: {}".format(total, has_title, total - has_title))

# === TACH SECTIONS ===
print("[3/6] Tach sections...")

def split_sections(text):
    if not text:
        return Row(introduction="", body="")
    intro_match = re.search(
        r'\n(?:1\.?\s*|I\.?\s*)?(?:Introduction|INTRODUCTION)\s*\n', text)
    if not intro_match:
        return Row(introduction="", body=text.strip())
    intro_start = intro_match.end()
    next_sec = re.compile(
        r'\n(?:\d+\.?\s+[A-Z][a-z]|II\.?\s+[A-Z]|(?:RELATED WORK|BACKGROUND|'
        r'METHODOLOGY|PROPOSED METHOD|LITERATURE REVIEW|PRELIMINARIES|'
        r'CONCLUSION|MODEL|EXPERIMENT|RESULTS|DISCUSSION|METHODS)\s*\n)')
    m = next_sec.search(text, intro_start)
    if m:
        return Row(introduction=text[intro_start:m.start()].strip(),
                   body=text[m.start():].strip())
    return Row(introduction=text[intro_start:].strip(), body="")

section_schema = StructType([
    StructField("introduction", StringType(), True),
    StructField("body", StringType(), True),
])
split_udf = udf(split_sections, section_schema)
df_joined = df_joined.withColumn("sections", split_udf(col("clean_text")))

# === CHUAN HOA SCHEMA ===
print("[4/6] Chuan hoa schema...")
df_final = df_joined.select(
    element_at(split(col("file_path"), "/"), -1).alias("paper_id"),
    "file_path", "raw_text", "clean_text",
    "abstract",
    col("sections.introduction").alias("introduction"),
    col("sections.body").alias("body"),
    "word_count_raw", "word_count_clean",
    "title", "authors", "categories", "quality_flag",
)

# === LUU ===
print("[5/6] Dem records...")
total_records = df_final.count()

print("[6/6] Ghi GCS...")
output = "gs://{}/silver/arxiv_silver_plus/".format(BUCKET)
df_final.write.mode("overwrite").parquet(output)

print("Hoan thanh! {} records -> {}".format(total_records, output))
for c in ["title", "authors", "categories", "abstract", "introduction"]:
    nc = df_final.filter(col(c).isNull()).count()
    print("  {}: {}/{} NULL ({:.1f}%)".format(c, nc, total_records, 100.0*nc/total_records))

spark.stop()


# **MINHASH_LSH ĐỂ PHÂN TÍCH DATASET (ĐÂY LÀ THEO HƯỚNG 1, KHÔNG CHẠY TRÊN COLAB, CHẠY TRÊN CLUSTER)**

In [ ]:
"""
04a_minhash_lsh.py - MinHash LSH Candidates (Offline)
Chay tren: Dataproc Cluster
Submit: gcloud dataproc jobs submit pyspark gs://BUCKET/scripts/04a_minhash_lsh.py \
            --cluster=bigdata-n9-ptit --region=asia-southeast1

PySpark ML hash + groupBy (KHONG dung approxSimilarityJoin)
Output: gs://BUCKET/intermediate/lsh_candidates/ (HIGH / MEDIUM)
"""

import re as re_module
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, BooleanType, ArrayType, StructType, StructField, DoubleType
from pyspark.ml.feature import CountVectorizer, NGram, RegexTokenizer, MinHashLSH
from pyspark.ml.linalg import SparseVector, DenseVector

BUCKET = "bigdata-n9-ptit-final"
INPUT = "gs://{}/silver/arxiv_silver_plus/".format(BUCKET)
OUTPUT = "gs://{}/intermediate/lsh_candidates/".format(BUCKET)

SHINGLE_K = 5
NUM_HASH_TABLES = 5
HIGH_THRESHOLD = 0.7
MEDIUM_THRESHOLD = 0.5

spark = SparkSession.builder \
    .appName("MinHash_LSH") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.memoryOverhead", "1g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Doc + chuan bi text
print("[1/7] Doc du lieu...")
df = spark.read.parquet(INPUT)
df_text = df.select("paper_id", F.coalesce(
    F.concat_ws(" ", F.col("abstract"), F.col("introduction"), F.col("body")),
    F.col("clean_text")).alias("text")
).filter(F.col("text").isNotNull() & (F.length(F.col("text")) > 200))
print("  So bai: {}".format(df_text.count()))

# PySpark ML pipeline
print("[2/7] Tokenize + NGram + CountVectorizer...")
tokenizer = RegexTokenizer(inputCol="text", outputCol="words", pattern="\\W+", minTokenLength=2)
df_words = tokenizer.transform(df_text).filter(F.size("words") >= 20)
ngram = NGram(n=SHINGLE_K, inputCol="words", outputCol="shingles")
df_shingles = ngram.transform(df_words).filter(F.size("shingles") > 0)
cv = CountVectorizer(inputCol="shingles", outputCol="features", minDF=2.0, vocabSize=1 << 18)
cv_model = cv.fit(df_shingles)
df_features = cv_model.transform(df_shingles)

@F.udf(BooleanType())
def has_nonzero(v):
    if v is None: return False
    if isinstance(v, SparseVector): return len(v.indices) > 0
    return True

df_features = df_features.filter(has_nonzero("features"))
print("  So bai hop le: {}".format(df_features.count()))

# MinHash transform (khong dung approxSimilarityJoin)
print("[3/7] MinHash transform...")
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=NUM_HASH_TABLES)
mh_model = mh.fit(df_features)
df_hashed = mh_model.transform(df_features).select("paper_id", "hashes").cache()
print("  Da hash: {}".format(df_hashed.count()))

# GroupBy band hash
print("[4/7] Tim candidates...")

@F.udf(StringType())
def extract_hash_val(band_idx, hash_vec):
    if hash_vec is None: return None
    return "b{}_{}".format(band_idx, int(hash_vec[0]))

df_bands = df_hashed.select("paper_id", F.posexplode("hashes").alias("band_idx", "hash_vec"))
df_bands = df_bands.withColumn("band_hash", extract_hash_val("band_idx", "hash_vec")).select("paper_id", "band_hash")

df_groups = df_bands.groupBy("band_hash").agg(F.collect_set("paper_id").alias("papers")) \
    .filter((F.size("papers") >= 2) & (F.size("papers") <= 100))

@F.udf(ArrayType(StructType([StructField("p1", StringType()), StructField("p2", StringType())])))
def make_pairs(papers):
    papers = sorted(papers)
    return [{"p1": papers[i], "p2": papers[j]} for i in range(len(papers)) for j in range(i+1, len(papers))]

df_pairs = df_groups.withColumn("pairs", make_pairs("papers"))
df_pairs = df_pairs.select(F.explode("pairs").alias("pair"))
df_pairs = df_pairs.select(F.col("pair.p1").alias("paper_a"), F.col("pair.p2").alias("paper_b")).distinct()
print("  Candidates: {}".format(df_pairs.count()))

# Tinh Jaccard tu hash agreement
print("[5/7] Tinh Jaccard...")
df_sim = df_pairs \
    .join(df_hashed.withColumnRenamed("paper_id","paper_a").withColumnRenamed("hashes","ha"), on="paper_a") \
    .join(df_hashed.withColumnRenamed("paper_id","paper_b").withColumnRenamed("hashes","hb"), on="paper_b")

@F.udf(DoubleType())
def hash_jaccard(ha, hb):
    if not ha or not hb: return 0.0
    return float(sum(1 for a, b in zip(ha, hb) if a[0] == b[0])) / len(ha)

df_sim = df_sim.withColumn("jaccard_similarity", hash_jaccard("ha", "hb"))
df_sim = df_sim.filter(F.col("jaccard_similarity") >= MEDIUM_THRESHOLD).select("paper_a", "paper_b", "jaccard_similarity")

# Loc cap cung arxiv ID
print("[6/7] Loc...")

@F.udf(StringType())
def base_id(pid):
    if not pid: return None
    return re_module.sub(r'v\d+$', '', pid.replace("arxiv_", "").replace(".pdf", ""))

df_sim = df_sim.withColumn("ba", base_id("paper_a")).withColumn("bb", base_id("paper_b"))
df_sim = df_sim.filter(F.col("ba") != F.col("bb")).drop("ba", "bb")
print("  Sau loc: {}".format(df_sim.count()))

# Phan loai + luu
print("[7/7] Luu...")
result = df_sim.withColumn("similarity_level",
    F.when(F.col("jaccard_similarity") >= HIGH_THRESHOLD, "HIGH").otherwise("MEDIUM")
).select("paper_a", "paper_b", "jaccard_similarity", "similarity_level")

result.groupBy("similarity_level").count().show()
result.write.partitionBy("similarity_level").mode("overwrite").parquet(OUTPUT)
result.filter(F.col("similarity_level") == "HIGH").orderBy(F.desc("jaccard_similarity")).show(10, truncate=False)

spark.stop()
print("Hoan thanh! -> {}".format(OUTPUT))


# **BUILD MINHASH_SIGN(CHẠY TRÊN CLUSTER)**

In [ ]:
"""
04b_minhash_signatures.py - Tao MinHash signatures (PySpark song song)
Chay tren: Dataproc Cluster
Submit: gcloud dataproc jobs submit pyspark gs://BUCKET/scripts/04b_minhash_signatures.py \
            --cluster=bigdata-n9-ptit --region=asia-southeast1

Output: gs://BUCKET/intermediate/minhash_signatures/ (paper_id, signature[128])
"""

import re as re_module
import hashlib
import random
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, IntegerType

BUCKET = "bigdata-n9-ptit-final"
INPUT = "gs://{}/silver/arxiv_silver_plus/".format(BUCKET)
OUTPUT = "gs://{}/intermediate/minhash_signatures/".format(BUCKET)
SHINGLE_K = 5
NUM_PERM = 128
LARGE_PRIME = 2147483647

# Hash params co dinh
random.seed(42)
HASH_A = [random.randint(1, LARGE_PRIME - 1) for _ in range(NUM_PERM)]
HASH_B = [random.randint(0, LARGE_PRIME - 1) for _ in range(NUM_PERM)]

spark = SparkSession.builder.appName("MinHash_Signatures") \
    .config("spark.sql.shuffle.partitions", "200").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

bc_a = spark.sparkContext.broadcast(HASH_A)
bc_b = spark.sparkContext.broadcast(HASH_B)
bc_p = spark.sparkContext.broadcast(LARGE_PRIME)
bc_k = spark.sparkContext.broadcast(SHINGLE_K)
bc_n = spark.sparkContext.broadcast(NUM_PERM)

print("[1/3] Doc du lieu...")
df = spark.read.parquet(INPUT)
df_text = df.select("paper_id", F.coalesce(
    F.concat_ws(" ", F.col("abstract"), F.col("introduction"), F.col("body")),
    F.col("clean_text")).alias("text")
).filter(F.col("text").isNotNull() & (F.length(F.col("text")) > 200))
print("  {} bai".format(df_text.count()))

print("[2/3] Tinh signatures (song song)...")

@F.udf(ArrayType(IntegerType()))
def compute_sig(text):
    if not text: return None
    k = bc_k.value
    words = re_module.findall(r'\w{2,}', text.lower())
    if len(words) < k: return None
    hashes = set()
    for i in range(len(words) - k + 1):
        s = " ".join(words[i:i+k])
        hashes.add(int(hashlib.md5(s.encode()).hexdigest(), 16) % bc_p.value)
    if not hashes: return None
    a, b, p, n = bc_a.value, bc_b.value, bc_p.value, bc_n.value
    sig = []
    for i in range(n):
        mv = p
        for h in hashes:
            v = (a[i] * h + b[i]) % p
            if v < mv: mv = v
        sig.append(mv)
    return sig

df_sigs = df_text.withColumn("signature", compute_sig("text"))
df_sigs = df_sigs.filter(F.col("signature").isNotNull()).select("paper_id", "signature")
print("  {} signatures".format(df_sigs.count()))

print("[3/3] Luu...")
df_sigs.write.mode("overwrite").parquet(OUTPUT)
spark.stop()
print("Hoan thanh! -> {}".format(OUTPUT))


# **BUILD LSH_INDEX**

In [ ]:
"""
04c_build_lsh_index.py - Build datasketch LSH index

Input:  gs://BUCKET/intermediate/minhash_signatures/
Output: gs://BUCKET/intermediate/minhash_index/
        - lsh_index.pkl, minhashes.pkl, config.json
"""

import os, time, pickle, json
import pandas as pd
import numpy as np
from datasketch import MinHash, MinHashLSH

BUCKET = "bigdata-n9-ptit-final"
SIGS_LOCAL = "/tmp/minhash_sigs/"
OUTPUT_LOCAL = "/tmp/minhash_index/"
GCS_OUTPUT = "gs://{}/intermediate/minhash_index/".format(BUCKET)
NUM_PERM = 128
LSH_THRESHOLD = 0.5

# === [1/4] Download signatures ===
print("[1/4] Download signatures tu GCS...")
t0 = time.time()
os.makedirs(SIGS_LOCAL, exist_ok=True)
os.system("gsutil -m cp 'gs://{}/intermediate/minhash_signatures/*.parquet' {}".format(BUCKET, SIGS_LOCAL))

df = pd.read_parquet(SIGS_LOCAL)
print("  {} bai, {:.0f}s".format(len(df), time.time() - t0))

# === [2/4] Tao MinHash objects ===
print("\n[2/4] Tao MinHash objects...")
t1 = time.time()
minhashes = {}
for i, row in df.iterrows():
    if i % 50000 == 0:
        print("  {}/{}".format(i, len(df)))
    m = MinHash(num_perm=NUM_PERM)
    m.hashvalues = np.array(row["signature"], dtype=np.uint64)
    minhashes[row["paper_id"]] = m
print("  {} objects, {:.0f}s".format(len(minhashes), time.time() - t1))

# === [3/4] Build LSH index ===
print("\n[3/4] Build LSH index (threshold={})...".format(LSH_THRESHOLD))
t2 = time.time()
lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=NUM_PERM)
for pid, mh in minhashes.items():
    lsh.insert(pid, mh)
print("  {:.0f}s".format(time.time() - t2))

# Test
test_pid = list(minhashes.keys())[0]
result = lsh.query(minhashes[test_pid])
print("\n  Test: {} -> {} matches".format(test_pid, len(result)))
for r in sorted(result)[:5]:
    if r != test_pid:
        sim = minhashes[test_pid].jaccard(minhashes[r])
        print("    {} | Jaccard={:.4f}".format(r, sim))

# === [4/4] Save + upload GCS ===
print("\n[4/4] Luu va upload GCS...")
os.makedirs(OUTPUT_LOCAL, exist_ok=True)

lsh_path = os.path.join(OUTPUT_LOCAL, "lsh_index.pkl")
with open(lsh_path, "wb") as f:
    pickle.dump(lsh, f)
print("  lsh_index.pkl: {:.1f} MB".format(os.path.getsize(lsh_path) / 1024 / 1024))

mh_path = os.path.join(OUTPUT_LOCAL, "minhashes.pkl")
with open(mh_path, "wb") as f:
    pickle.dump(minhashes, f)
print("  minhashes.pkl: {:.1f} MB".format(os.path.getsize(mh_path) / 1024 / 1024))

with open(os.path.join(OUTPUT_LOCAL, "config.json"), "w") as f:
    json.dump({"num_perm": NUM_PERM, "threshold": LSH_THRESHOLD, "n_papers": len(minhashes)}, f)

os.system("gsutil -m cp {}* {}".format(OUTPUT_LOCAL, GCS_OUTPUT))

print("\n[HOAN THANH] {:.0f} phut | {} papers -> {}".format(
    (time.time() - t0) / 60, len(minhashes), GCS_OUTPUT))

[1/4] Download signatures tu GCS...
  1000 bai, 9s

[2/4] Tao MinHash objects...
  0/1000
  1000 objects, 1s

[3/4] Build LSH index (threshold=0.5)...
  0s

  Test: arxiv_0803.3266.pdf -> 1 matches

[4/4] Luu va upload GCS...
  lsh_index.pkl: 1.4 MB
  minhashes.pkl: 3.1 MB

[HOAN THANH] 0 phut | 1000 papers -> gs://bigdata-n9-ptit-final/intermediate/minhash_index/


# **build FAISS_index**

In [ ]:
"""
04e_chunk_export.py - Export text chunks (PySpark)
Chay tren: Dataproc Cluster
Submit: gcloud dataproc jobs submit pyspark gs://BUCKET/scripts/04e_chunk_export.py \
            --cluster=bigdata-n9-ptit --region=asia-southeast1

Input:  gs://BUCKET/silver/arxiv_silver_plus/
Output: gs://BUCKET/intermediate/chunks_parquet/
Schema: paper_id, chunk_id, section, chunk_text, chunk_uid
"""

import re as re_module
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType

BUCKET = "bigdata-n9-ptit-final"
INPUT = "gs://{}/silver/arxiv_silver_plus/".format(BUCKET)
OUTPUT = "gs://{}/intermediate/chunks_parquet/".format(BUCKET)
CHUNK_SIZE = 4

spark = SparkSession.builder.appName("Chunk_Export") \
    .config("spark.sql.shuffle.partitions", "200").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("[1/3] Doc du lieu...")
df = spark.read.parquet(INPUT)
print("  {} bai".format(df.count()))

chunk_schema = ArrayType(StructType([
    StructField("chunk_id", IntegerType(), False),
    StructField("section", StringType(), True),
    StructField("chunk_text", StringType(), False),
]))

@F.udf(chunk_schema)
def create_chunks(abstract, introduction, body):
    chunks, cid = [], 0
    for sec, text in [("abstract", abstract or ""), ("introduction", introduction or ""), ("body", body or "")]:
        if len(text.strip()) < 50: continue
        sents = [s.strip() for s in re_module.split(r'(?<=[.!?])\s+', text.strip()) if len(s.strip()) >= 20]
        for i in range(0, len(sents), CHUNK_SIZE):
            ct = " ".join(sents[i:i+CHUNK_SIZE]).strip()
            if len(ct) >= 50:
                chunks.append({"chunk_id": cid, "section": sec, "chunk_text": ct})
                cid += 1
    return chunks

print("[2/3] Tao chunks...")
df_out = df.select("paper_id", "abstract", "introduction", "body") \
    .withColumn("chunks", create_chunks("abstract", "introduction", "body"))
df_out = df_out.select("paper_id", F.explode("chunks").alias("c"))
df_out = df_out.select("paper_id", F.col("c.chunk_id").alias("chunk_id"),
    F.col("c.section").alias("section"), F.col("c.chunk_text").alias("chunk_text"))
df_out = df_out.withColumn("chunk_uid", F.concat_ws("_", "paper_id", F.col("chunk_id").cast("string")))

total = df_out.count()
print("  {} chunks".format(total))
df_out.groupBy("section").count().show()

print("[3/3] Luu...")
df_out.write.mode("overwrite").parquet(OUTPUT)
spark.stop()
print("Hoan thanh! {} chunks -> {}".format(total, OUTPUT))


In [ ]:
import os, json, time, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.fs as pafs

BUCKET = "bigdata-n9-ptit-final"
CHUNKS_INPUT = "{}/intermediate/chunks_parquet".format(BUCKET)
MODEL_NAME = "allenai/specter"
EMBEDDING_DIM = 768
BATCH_SIZE = 128
SECTIONS_TO_EMBED = None
EMB_FILE = "/tmp/embeddings.npy"
META_FILE = "/tmp/chunk_metadata.parquet"

gcs = pafs.GcsFileSystem()

# === [1/6] Doc chunks - chi cot can thiet ===
print("[1/6] Doc chunks...")
t0 = time.time()

filters = [("section", "in", SECTIONS_TO_EMBED)] if SECTIONS_TO_EMBED else None
table = pq.read_table(
    CHUNKS_INPUT,
    filesystem=gcs,
    columns=["paper_id", "chunk_id", "section", "chunk_text"],
    filters=filters
)
df_chunks = table.to_pandas()
del table; gc.collect()

print("  Tong sau loc: {}".format(len(df_chunks)))

df_chunks = df_chunks.dropna(subset=["chunk_text"])
df_chunks = df_chunks[df_chunks["chunk_text"].str.len() > 20].reset_index(drop=True)
n_total = len(df_chunks)
print("  {} chunks, {:.0f}s".format(n_total, time.time() - t0))

df_chunks["index_position"] = range(n_total)
df_chunks[["paper_id", "chunk_id", "section", "chunk_text", "index_position"]].to_parquet(META_FILE, index=False)

del df_chunks; gc.collect()
print("  Giai phong RAM metadata")

# === [2/6] Load model ===
print("\n[2/6] Load model...")
import torch
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("  Device: {}".format(device))

# === [3/6] Embed - ghi ra disk bang memmap ===
print("\n[3/6] Embed {} chunks (memmap)...".format(n_total))
t1 = time.time()
emb_mmap = np.memmap(EMB_FILE, dtype=np.float32, mode='w+', shape=(n_total, EMBEDDING_DIM))

total_b = (n_total + BATCH_SIZE - 1) // BATCH_SIZE
parquet_file = pq.ParquetFile(META_FILE)

current_idx = 0
bn = 0

for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE, columns=["chunk_text"]):
    bn += 1
    batch_texts = batch.column("chunk_text").to_pylist()
    actual_batch_size = len(batch_texts)

    inputs = tokenizer(batch_texts, padding=True, truncation=True,
                       max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs)

    emb = out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)
    emb_mmap[current_idx : current_idx + actual_batch_size] = emb
    current_idx += actual_batch_size

    del inputs, out, emb, batch_texts
    if device == "cuda" and bn % 50 == 0:
        torch.cuda.empty_cache()
        gc.collect()

    if bn % 200 == 0 or bn == 1 or bn == total_b:
        el = time.time() - t1
        sp = current_idx / el if el > 0 else 0
        eta = (n_total - current_idx) / sp / 60 if sp > 0 else 0
        print("  {}/{} | {:.0f}/s | ETA {:.0f}m".format(bn, total_b, sp, eta))

emb_mmap.flush()
del model, tokenizer, parquet_file
if device == "cuda": torch.cuda.empty_cache()
gc.collect()
print("  {:.0f} phut | Giai phong model".format((time.time() - t1) / 60))

# === [4/6] Normalize + Build FAISS ===
print("\n[4/6] Build FAISS index...")
import faiss

emb_rw = np.memmap(EMB_FILE, dtype=np.float32, mode='r+', shape=(n_total, EMBEDDING_DIM))
NORM_BATCH = 50000
for s in range(0, n_total, NORM_BATCH):
    e = min(s + NORM_BATCH, n_total)
    chunk = np.array(emb_rw[s:e])
    faiss.normalize_L2(chunk)
    emb_rw[s:e] = chunk
    del chunk
emb_rw.flush()
del emb_rw

embeddings = np.memmap(EMB_FILE, dtype=np.float32, mode='r', shape=(n_total, EMBEDDING_DIM))
if n_total < 100000:
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    idx_type = "IndexFlatIP"
else:
    nlist = min(int(np.sqrt(n_total)), 4096)
    quantizer = faiss.IndexFlatIP(EMBEDDING_DIM)
    index = faiss.IndexIVFFlat(quantizer, EMBEDDING_DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    train_data = np.array(embeddings[:min(n_total, 100000)])
    index.train(train_data)
    del train_data
    idx_type = "IndexIVFFlat(nlist={})".format(nlist)

ADD_BATCH = 50000
for s in range(0, n_total, ADD_BATCH):
    e = min(s + ADD_BATCH, n_total)
    index.add(np.array(embeddings[s:e]))
    print("  Added {}/{}".format(e, n_total))
print("  {} | {} vectors".format(idx_type, index.ntotal))

# === [5/6] Test ===
print("\n[5/6] Test...")
df_meta = pd.read_parquet(META_FILE)
q = np.array(embeddings[0:1])
d, idx = index.search(q, 5)
for r, (dist, ix) in enumerate(zip(d[0], idx[0])):
    row = df_meta.iloc[ix]
    print("  [{}] {:.4f} | {} | {}...".format(r+1, dist, row["paper_id"], str(row["chunk_text"])[:60]))

# === [6/6] Upload ===
print("\n[6/6] Luu GCS...")
faiss.write_index(index, "/tmp/faiss_index.bin")
config = {"model": MODEL_NAME, "dim": EMBEDDING_DIM, "n": n_total,
          "type": idx_type, "sections": SECTIONS_TO_EMBED}
with open("/tmp/index_config.json", "w") as f: json.dump(config, f)

os.system("gsutil cp /tmp/faiss_index.bin gs://{}/intermediate/faiss_index/".format(BUCKET))
os.system("gsutil cp {} gs://{}/intermediate/faiss_index/".format(META_FILE, BUCKET))
os.system("gsutil cp /tmp/index_config.json gs://{}/intermediate/faiss_index/".format(BUCKET))

os.remove(EMB_FILE)
print("Hoan thanh! {:.0f} phut | {} vectors".format((time.time() - t0) / 60, n_total))

[1/6] Doc chunks...
  Tong: 76389 chunks
  Sau loc ['abstract']: 1670
  7s

[2/6] Load model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Device: cpu

[3/6] Embed 1670 chunks...
  1/14 | 0/s | ETA 0m


KeyboardInterrupt: 

# **DEMO**

In [ ]:
"""
05_demo_check_pdf.py - Demo phat hien dao van (Realtime)
Chay tren: Colab (GPU)
"""
import hashlib, random
import os, re, time, pickle, unicodedata
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.fs as pafs

BUCKET = "bigdata-n9-ptit-final"
SHINGLE_K = 5
NUM_PERM = 128
FUZZY_THRESHOLD = 0.85
SEMANTIC_THRESHOLD = 0.85

gcs = pafs.GcsFileSystem()

# =================================================================
# LOAD DU LIEU (1 lan)
# =================================================================
print("[1/4] Load metadata...")
t0 = time.time()
df_meta = pq.read_table("bigdata-n9-ptit-final/silver/arxiv_silver_plus", filesystem=gcs,
    columns=["paper_id", "title", "authors", "categories"]).to_pandas()
meta_dict = {row["paper_id"]: row.to_dict() for _, row in df_meta.iterrows()}
print("  {} bai".format(len(meta_dict)))

def pid_to_url(pid):
    base = re.sub(r'v\d+$', '', pid.replace("arxiv_", "").replace(".pdf", ""))
    return "https://arxiv.org/abs/{}".format(base)

print("[2/4] Load FAISS...")
os.makedirs("/tmp/faiss", exist_ok=True)
if not os.path.exists("/tmp/faiss/faiss_index.bin"):
    os.system("gsutil cp gs://{}/intermediate/faiss_index/faiss_index.bin /tmp/faiss/faiss_index.bin".format(BUCKET))
import faiss
faiss_index = faiss.read_index("/tmp/faiss/faiss_index.bin")
df_faiss_meta = pq.read_table("bigdata-n9-ptit-final/intermediate/faiss_index/chunk_metadata.parquet",
    filesystem=gcs).to_pandas()
print("  {} vectors".format(faiss_index.ntotal))

print("[3/4] Load MinHash LSH...")
os.makedirs("/tmp/minhash", exist_ok=True)
for f in ["lsh_index.pkl", "minhashes.pkl"]:
    if not os.path.exists("/tmp/minhash/" + f):
        os.system("gsutil cp gs://{}/intermediate/minhash_index/{} /tmp/minhash/".format(BUCKET, f))
with open("/tmp/minhash/lsh_index.pkl", "rb") as f: lsh_index = pickle.load(f)
with open("/tmp/minhash/minhashes.pkl", "rb") as f: minhashes_dict = pickle.load(f)
print("  {} papers".format(len(minhashes_dict)))

print("[4/4] Load model...")
import torch
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("allenai/specter")
emb_model = AutoModel.from_pretrained("allenai/specter")
device = "cuda" if torch.cuda.is_available() else "cpu"
emb_model = emb_model.to(device).eval()
print("  Device: {}".format(device))

try:
    from Levenshtein import ratio as lev_ratio
except ImportError:
    from difflib import SequenceMatcher
    def lev_ratio(s1, s2): return SequenceMatcher(None, s1, s2).ratio()

print("San sang! {:.0f}s\n".format(time.time() - t0))

# =================================================================
# TIEN XU LY (DONG BO VOI 02_clean_text.py)
# =================================================================
import fitz
from datasketch import MinHash


def extract_text_from_pdf(fp):
    doc = fitz.open(fp)
    text = "\n".join(p.get_text() for p in doc).strip()
    doc.close()
    return text

def clean_text(text):
    """9 tang lam sach - DONG BO voi 02_clean_text.py"""
    if not text: return ""
    t = text
    # Tang 1: ArXiv header
    t = re.sub(r'^arXiv:\S+\s+\[[\w.\-]+\]\s+\d+\s+\w+\s+\d{4}\s*\n', '', t, flags=re.MULTILINE)
    # Tang 2: Journal metadata
    t = re.sub(
        r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|'
        r'Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
        r'\s+\d{1,2},?\s+\d{4}\n\d{1,2}:\d{2}\n', '', t)
    t = re.sub(
        r'(?:WSPC|World Scientific|Typeset with|Preprint submitted to|'
        r'Preprint typeset using|Draft version)[^\n]*\n(?:[^\n]{0,60}\n){0,2}', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^[A-Z]{2,10}-[\w/\-]+(?:,\s*[A-Z]{2,10}-[\w/\-]+)*\s*\n', '', t, flags=re.MULTILINE)
    # Tang 3: References (chi 40% cuoi)
    total_len = len(t)
    ss = int(total_len * 0.60)
    rm = re.search(r'\n\s*(?:References|REFERENCES|Bibliography|BIBLIOGRAPHY)\s*\n', t[ss:])
    if rm: t = t[:ss + rm.start()]
    else:
        ss2 = int(total_len * 0.75)
        rm2 = re.search(r'\n(?:\[\d+\]|\d+[\)\.]\s)\s*[A-Z][^.]{15,}\n(?:(?:\[\d+\]|\d+[\)\.]\s)\s*.{10,}\n){4,}', t[ss2:])
        if rm2: t = t[:ss2 + rm2.start()]
    # Tang 4: So trang
    t = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', t)
    t = re.sub(r'\n\s*\d+/\d+\s*\n', '\n', t)
    t = re.sub(r'\bpage\s+\d+\s+of\s+\d+\b', '', t, flags=re.IGNORECASE)
    # Tang 5: Ligatures
    for lig, rep in {'\ufb00':'ff','\ufb01':'fi','\ufb02':'fl',
                     '\ufb03':'ffi','\ufb04':'ffl','\ufb05':'st','\ufb06':'st'}.items():
        t = t.replace(lig, rep)
    # Tang 6: Unicode NFKC
    t = unicodedata.normalize("NFKC", t)
    # Tang 7: Ky tu khong in
    t = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', t)
    # Tang 8: Hyphenation
    t = re.sub(r'(\w)-\n(\w)', r'\1\2', t)
    # Tang 9: Email, URL, whitespace
    t = re.sub(r'\[?\w+@[\w.]+\]?\(?mailto:[\w@.]+\)?', '', t)
    t = re.sub(r'\S+@\S+\.\S+', '', t)
    t = re.sub(r'https?://\S+', '', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[ \t]{2,}', ' ', t)
    return t.strip()

def extract_abstract(text):
    m = re.search(r'(?:abstract|ABSTRACT)\s*[:\.]?\s*\n?(.*?)(?:\n\s*(?:1\.?\s*|I\.?\s*)?'
                  r'(?:Introduction|INTRODUCTION|Keywords|KEYWORDS))', text, re.DOTALL|re.IGNORECASE)
    if m and len(m.group(1).strip()) > 50: return m.group(1).strip()
    m2 = re.search(r'(?:abstract|ABSTRACT)\s*[:\.]?\s*\n?(.{100,2000})', text, re.DOTALL|re.IGNORECASE)
    if m2: return m2.group(1).strip()
    return " ".join(text.split()[:300])

def split_sentences(text):
    if not text or len(str(text)) < 30: return []
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(text).strip()) if len(s.strip()) >= 30]

def embed_text(text):
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad(): out = emb_model(**inputs)
    emb = out.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32)
    faiss.normalize_L2(emb)
    return emb

random.seed(42)
LARGE_PRIME = 2147483647
HASH_A = [random.randint(1, LARGE_PRIME - 1) for _ in range(NUM_PERM)]
HASH_B = [random.randint(0, LARGE_PRIME - 1) for _ in range(NUM_PERM)]

def text_to_minhash(text):
    """Tao MinHash CUNG hash params voi 04b_minhash_signatures.py"""
    words = re.findall(r'\w{2,}', text.lower())
    if len(words) < SHINGLE_K: return None

    # Hash shingles
    shingle_hashes = set()
    for i in range(len(words) - SHINGLE_K + 1):
        s = " ".join(words[i:i+SHINGLE_K])
        h = int(hashlib.md5(s.encode()).hexdigest(), 16) % LARGE_PRIME
        shingle_hashes.add(h)

    if not shingle_hashes: return None

    sig = []
    for i in range(NUM_PERM):
        a, b = HASH_A[i], HASH_B[i]
        min_val = LARGE_PRIME
        for h in shingle_hashes:
            val = (a * h + b) % LARGE_PRIME
            if val < min_val: min_val = val
        sig.append(min_val)

    # Dat vao datasketch MinHash object
    m = MinHash(num_perm=NUM_PERM)
    m.hashvalues = np.array(sig, dtype=np.uint64)
    return m

# =================================================================
# 5 LAYERS
# =================================================================
def layer1_minhash(query_mh):
    if query_mh is None: return 0.0, {}
    candidates = lsh_index.query(query_mh)
    results = {}
    for pid in candidates:
        if pid in minhashes_dict:
            sim = query_mh.jaccard(minhashes_dict[pid])
            if sim >= 0.3: results[pid] = {"jaccard": round(sim, 4)}
    return round(max((v["jaccard"] for v in results.values()), default=0.0), 4), results

def layer2_fuzzy(query_sents, candidate_pids):
    pid_list = list(candidate_pids)
    try:
        df_text = pq.read_table("bigdata-n9-ptit-final/silver/arxiv_silver_plus", filesystem=gcs,
            columns=["paper_id","abstract","introduction","body"],
            filters=[("paper_id","in",pid_list)]).to_pandas()
        text_dict = {row["paper_id"]: row.to_dict() for _, row in df_text.iterrows()}
    except: text_dict = {}

    q_sorted = sorted(query_sents, key=len, reverse=True)[:100]
    results = {}
    for pid in pid_list:
        paper = text_dict.get(pid)
        if not paper: continue
        other_sents = []
        for sec in ["abstract","introduction","body"]:
            other_sents.extend(split_sentences(paper.get(sec, "")))
        if not other_sents: continue
        fc = 0
        for qs in q_sorted:
            ql = qs.lower()
            for os_ in other_sents:
                if max(len(qs),len(os_)) > 2.5 * min(len(qs),len(os_)): continue
                if lev_ratio(ql, os_.lower()) >= FUZZY_THRESHOLD:
                    fc += 1; break
        if fc > 0:
            results[pid] = {"fuzzy_count": fc, "fuzzy_ratio": round(fc/max(len(q_sorted),1), 4)}
    return round(max((v["fuzzy_ratio"] for v in results.values()), default=0.0), 4), results

def layer3_semantic(query_emb, top_k=20):
    dists, idxs = faiss_index.search(query_emb, top_k)
    results = {}
    for d, i in zip(dists[0], idxs[0]):
        if i < 0 or i >= len(df_faiss_meta): continue
        pid = df_faiss_meta.iloc[i]["paper_id"]
        sim = float(d)
        if pid not in results or sim > results[pid]["cosine"]:
            results[pid] = {"cosine": round(sim, 4)}
    ms = max((v["cosine"] for v in results.values()), default=0.0)
    return round(max(0, (ms - SEMANTIC_THRESHOLD) / (1 - SEMANTIC_THRESHOLD)), 4), results

def layer4_concept(qtitle, qabs, candidate_pids):
    from difflib import SequenceMatcher
    cp = [r'we propose',r'we present',r'we introduce',r'novel',r'our method',r'our approach']
    qc = set(p for p in cp if re.search(p, qabs.lower()))
    qn = set(re.findall(r'(\d+\.?\d*)\s*%', qabs))
    qw = set(re.findall(r'\b[a-z]{4,}\b', qabs.lower()))
    pid_list = list(candidate_pids)
    try:
        df_abs = pq.read_table("bigdata-n9-ptit-final/silver/arxiv_silver_plus", filesystem=gcs,
            columns=["paper_id","abstract"], filters=[("paper_id","in",pid_list)]).to_pandas()
        abs_dict = {r["paper_id"]: str(r["abstract"]) for _, r in df_abs.iterrows()}
    except: abs_dict = {}

    results = {}
    for pid in pid_list:
        info = meta_dict.get(pid, {})
        ot = str(info.get("title","")).lower()
        oa = abs_dict.get(pid, "")
        ts = SequenceMatcher(None, qtitle.lower(), ot).ratio()
        ow = set(re.findall(r'\b[a-z]{4,}\b', oa.lower()))
        ko = len(qw&ow)/max(len(qw|ow),1) if qw and ow else 0
        oc = set(p for p in cp if re.search(p, oa.lower()))
        cs = len(qc&oc)/max(len(qc|oc),1) if qc else 0
        on = set(re.findall(r'(\d+\.?\d*)\s*%', oa))
        ns = len(qn&on)/max(len(qn|on),1) if qn else 0
        sc = 0.3*ts + 0.3*ko + 0.2*cs + 0.2*ns
        if sc > 0.2: results[pid] = {"concept": round(sc, 4)}
    return round(max((v["concept"] for v in results.values()), default=0.0), 4), results

def layer5_self(q_authors, candidate_pids, all_scores):
    if not q_authors: return 0.0, {}
    qa = set(a.lower().strip() for a in q_authors)
    results = {}
    for pid in candidate_pids:
        info = meta_dict.get(pid, {})
        oa = set(a.lower().strip() for a in (info.get("authors") or []))
        common = qa & oa
        if common:
            results[pid] = {"common": list(common)[:3], "sim": round(all_scores.get(pid, 0), 4)}
    return round(max((v["sim"] for v in results.values()), default=0.0), 4), results

def scoring(l1, l2, l3, l4, l5):
    if l1 < 0.05: f = 0.35*l2 + 0.45*l3 + 0.10*l4 + 0.10*l5
    else: f = 0.40*l1 + 0.25*l2 + 0.25*l3 + 0.10*l4
    if l1 > 0.3: f *= 1.2
    if l2 > 0.5: f *= 1.15
    f = min(f, 1.0)
    conf = "HIGH" if l1 > 0.15 or l2 > 0.25 else "MEDIUM" if l3 > 0.5 else "LOW"
    if f > 0.65 and conf in ["HIGH","MEDIUM"]: v,d = "PLAGIARISM","Phat hien dao van nghiem trong"
    elif f > 0.50: v,d = "HIGHLY SUSPICIOUS","Nghi ngo cao"
    elif f > 0.35: v,d = "SUSPICIOUS","Nghi ngo, co doan tuong tu"
    elif f > 0.20: v,d = "SIMILAR","Tuong tu nhe"
    else: v,d = "CLEAN","Khong phat hien dao van"
    return {"score":round(f,4),"verdict":v,"desc":d,"confidence":conf,
            "self_plagiarism":"Co" if l5>0.5 else "Khong",
            "L1":round(l1,4),"L2":round(l2,4),"L3":round(l3,4),"L4":round(l4,4),"L5":round(l5,4)}

# =================================================================
# HAM CHINH
# =================================================================
def check_pdf(file_path, authors=None):
    print("=" * 60)
    print("KIEM TRA DAO VAN")
    print("=" * 60)
    t0 = time.time()

    print("\n[1/7] Extract + clean (9 tang)...")
    full_text = clean_text(extract_text_from_pdf(file_path))
    abstract = extract_abstract(full_text)
    query_sents = split_sentences(full_text)[:200]
    tm = re.match(r'(.{10,200}?)[\n\r]', full_text)
    qtitle = tm.group(1).strip() if tm else full_text[:100]
    print("  Title: {}".format(qtitle[:80]))
    print("  So cau: {}".format(len(query_sents)))

    print("\n[2/7] L1 - MinHash LSH...")
    t1 = time.time()
    qmh = text_to_minhash(full_text)
    l1_score, l1_res = layer1_minhash(qmh)
    print("  {} candidates | score={} | {:.1f}s".format(len(l1_res), l1_score, time.time()-t1))
    for pid in sorted(l1_res, key=lambda x: l1_res[x]["jaccard"], reverse=True)[:5]:
        print("    {} | J={} | {}".format(pid, l1_res[pid]["jaccard"], str(meta_dict.get(pid,{}).get("title",""))[:60]))

    print("\n[3/7] L3 - Semantic (FAISS)...")
    t2 = time.time()
    qemb = embed_text(abstract)
    l3_score, l3_res = layer3_semantic(qemb)
    print("  {} candidates | score={} | {:.1f}s".format(len(l3_res), l3_score, time.time()-t2))
    for pid in sorted(l3_res, key=lambda x: l3_res[x]["cosine"], reverse=True)[:5]:
        print("    {} | cos={} | {}".format(pid, l3_res[pid]["cosine"], str(meta_dict.get(pid,{}).get("title",""))[:60]))

    all_cands = set(l1_res.keys()) | set(l3_res.keys())
    print("\n  Union: {} (L1:{} + L3:{})".format(len(all_cands), len(l1_res), len(l3_res)))

    print("\n[4/7] L2 - Fuzzy match...")
    t3 = time.time()
    l2_score, l2_res = layer2_fuzzy(query_sents, all_cands)
    print("  {} matches | score={} | {:.1f}s".format(len(l2_res), l2_score, time.time()-t3))
    for pid in sorted(l2_res, key=lambda x: l2_res[x]["fuzzy_ratio"], reverse=True)[:3]:
        r = l2_res[pid]
        print("    {} | fuzzy={} ({}/{})".format(pid, r["fuzzy_ratio"], r["fuzzy_count"], len(query_sents)))

    print("\n[5/7] L4 - Concept match...")
    t4 = time.time()
    l4_score, l4_res = layer4_concept(qtitle, abstract, all_cands)
    print("  {} matches | score={} | {:.1f}s".format(len(l4_res), l4_score, time.time()-t4))

    print("\n[6/7] L5 - Self-plagiarism...")
    all_scores = {}
    for pid in all_cands:
        sc = []
        if pid in l1_res: sc.append(l1_res[pid]["jaccard"])
        if pid in l2_res: sc.append(l2_res[pid]["fuzzy_ratio"])
        if pid in l3_res: sc.append(l3_res[pid]["cosine"])
        all_scores[pid] = max(sc) if sc else 0
    l5_score, l5_res = layer5_self(authors or [], all_cands, all_scores)
    print("  {} matches | score={}".format(len(l5_res), l5_score))

    result = scoring(l1_score, l2_score, l3_score, l4_score, l5_score)
    tt = time.time() - t0

    print("\n[7/7] KET QUA")
    print("=" * 60)
    print("  Verdict:     {}".format(result["verdict"]))
    print("  Diem tong:   {:.1f}%".format(result["score"] * 100))
    print("  Do tin cay:  {}".format(result["confidence"]))
    print("  Self-plag:   {}".format(result["self_plagiarism"]))
    print("  L1={:.1f}%  L2={:.1f}%  L3={:.1f}%  L4={:.1f}%  L5={:.1f}%".format(
        result["L1"]*100, result["L2"]*100, result["L3"]*100, result["L4"]*100, result["L5"]*100))
    print("  Thoi gian:   {:.1f}s".format(tt))

    print("\n" + "-" * 60)
    print("DANH SACH BAI TUONG TU")
    print("-" * 60)
    ranked = sorted(all_cands, key=lambda p: all_scores.get(p, 0), reverse=True)
    for i, pid in enumerate(ranked[:15]):
        info = meta_dict.get(pid, {})
        sc = all_scores.get(pid, 0)
        l1v = l1_res.get(pid, {}).get("jaccard", 0)
        l2v = l2_res.get(pid, {}).get("fuzzy_ratio", 0)
        l3v = l3_res.get(pid, {}).get("cosine", 0)
        if sc >= 0.7: lv = "RAT CAO"
        elif sc >= 0.5: lv = "CAO"
        elif sc >= 0.3: lv = "TB"
        else: lv = "THAP"
        print("\n  [{}] {} | {}".format(i+1, lv, str(info.get("title",""))[:70]))
        print("      L1={:.0f}% L2={:.0f}% L3={:.0f}% Max={:.0f}%".format(l1v*100, l2v*100, l3v*100, sc*100))
        print("      {}".format(pid_to_url(pid)))

    return result

print("check_pdf() san sang. Su dung:")
print('  from google.colab import files')
print('  uploaded = files.upload()')
print('  result = check_pdf(list(uploaded.keys())[0])')


[1/4] Load metadata...
  306384 bai
[2/4] Load FAISS...
  574768 vectors
[3/4] Load MinHash LSH...
  306380 papers
[4/4] Load model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Device: cuda
San sang! 479s

check_pdf() san sang. Su dung:
  from google.colab import files
  uploaded = files.upload()
  result = check_pdf(list(uploaded.keys())[0])


In [ ]:
from google.colab import files

uploaded = files.upload()
result = check_pdf(list(uploaded.keys())[0])


Saving bronze_raw_pdfs_arxiv_arxiv_0704.0002.pdf to bronze_raw_pdfs_arxiv_arxiv_0704.0002.pdf
KIEM TRA DAO VAN

[1/7] Extract + clean (9 tang)...
  Title: Sparsity-certifying Graph Decompositions
  So cau: 200

[2/7] L1 - MinHash LSH...
  1 candidates | score=0.9922 | 0.4s
    arxiv_0704.0002.pdf | J=0.9922 | Sparsity-certifying Graph Decompositions

[3/7] L3 - Semantic (FAISS)...
  20 candidates | score=0.4113 | 0.6s
    arxiv_1302.5754.pdf | cos=0.9117 | Enumeration Based Search Algorithm For Finding A Regular Bi-
    arxiv_2405.10238.pdf | cos=0.9052 | Rounding Large Independent Sets on Expanders
    arxiv_2303.04590.pdf | cos=0.9048 | An Exploration of Graph Pebbling
    arxiv_2407.02638.pdf | cos=0.8991 | A Refutation of the Pach-Tardos Conjecture for 0-1 Matrices
    arxiv_2303.15367.pdf | cos=0.8982 | Uniformly Random Colourings of Sparse Graphs

  Union: 21 (L1:1 + L3:20)

[4/7] L2 - Fuzzy match...
  1 matches | score=1.0 | 498.2s
    arxiv_0704.0002.pdf | fuzzy=1.0 (100/200)

